# Smart microscopy 1: keep a moving object in view

A closed feedback loop: **snap → find the object → move the stage so it is centred → repeat.**
The stage path, together with where the object sat in each frame, records the object's trajectory
in µm. That trajectory is far longer than one field of view, e.g. for a swimming ciliate, a
crawling cell or a drifting colony.

Two ways to find the object:

- **`template`** follows whatever texture is in the centre of the first frame. It is the default
  and works for cells in tissue, organisms, or any structured target.
- **`blob`** follows the bright (or dark) spot closest to where the object is expected. Use it
  for isolated objects on an even background.

**Safety.** All moves are relative and X/Y only (Z is never moved). Each move is limited to
`MAX_STEP_UM` and to a box of `±BOX_HALF_WIDTH_UM` around the start, and a lost target stops the
loop. Interrupt the cell (■) to stop at any time.

**Without a moving sample** (e.g. the virtual microscope), set `SIMULATE_MOTION_UM`. The notebook
then adds a random walk to the stage, which looks to the camera as if the object moved. The tracker
should cancel it, and the measured object position should stay put.

## 1 Connect

In [ ]:
import json, time, datetime
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

import imswitchclient.ImSwitchClient as imc
from imswitchclient import smart

# None = $IMSWITCH_API_URL, set by ImSwitch for the notebooks it serves itself.
# From another computer use the browser address, e.g. "http://192.168.178.76/imswitch/api"
IMSWITCH_URL = None

client = imc.ImSwitchClient(IMSWITCH_URL)
print("Connected to:", client.base_uri)
if not hasattr(client, "autofocusManager"):
    raise ImportError("imswitchclient is too old (no smart helpers) - update it, e.g. pip install -U "
                      "https://github.com/openUC2/imswitchclient/archive/refs/heads/main.zip")

STAGE = client.positionersManager.allStageNames[0]
SPEED = 5000                                   # stage speed for all moves
snap = smart.make_snap(client)                 # -> float32 grayscale frame
move = smart.make_relative_mover(client, STAGE, speed=SPEED)   # move(dx_um, dy_um), blocking
WORK_DIR = Path.home() / "smart_microscopy" / "tracking"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Stage:", STAGE, client.positionersManager.getPositionerPositions()[STAGE])
print("Light sources:", client.lasersManager.getLaserNames())

## 2 Light and a first look

Switch on the illumination you image with. `ILLUMINATION = None` leaves it as it is (e.g. when
you set it up in the ImSwitch UI).

In [ ]:
ILLUMINATION = {"type": "laser", "name": "LED", "value": 1000}
# LED matrix instead:  ILLUMINATION = {"type": "ledmatrix", "intensity": 128}

def light(on=True):
    if ILLUMINATION is None:
        return
    if ILLUMINATION["type"] == "ledmatrix":
        if on:
            client.ledMatrixManager.setAllLED(state=1, intensity=ILLUMINATION["intensity"])
        else:
            client.ledMatrixManager.setAllLEDOff()
    elif on:
        client.lasersManager.setLaserOn(ILLUMINATION["name"], ILLUMINATION["value"])
    else:
        client.lasersManager.setLaserOff(ILLUMINATION["name"])

light(True)
time.sleep(0.5)
frame = snap()
H, W = frame.shape
print(f"frame {W} x {H} px, range {frame.min():.0f}..{frame.max():.0f}")
plt.figure(figsize=(6, 4.5)); plt.imshow(frame, cmap="gray"); plt.axis("off"); plt.show()

## 3 Camera ↔ stage calibration

Closed-loop moves convert "the object is 40 px left of the centre" into a stage move. The notebook
measures that conversion itself. It moves +X and +Y by a few percent of the field of view and finds
the image shift by phase correlation. The measurement includes camera rotation, mirroring and axis
signs, so the loop cannot run the wrong way. The stage returns to where it started.

Use a structured area of the sample. Recalibrate after changing objective, binning or camera. The
result is cached in `WORK_DIR`.

In [ ]:
RECALIBRATE = False
CAL_FILE = WORK_DIR / "stage_camera_calibration.json"


def pixel_size_guess():
    """µm/px from ImSwitch's pixel calibration or objective, 1.0 if neither is known"""
    try:
        r = client.get_json("/PixelCalibrationController/getPixelSizeUm")
        if r.get("pixelSizeUm"):
            return float(r["pixelSizeUm"])
    except Exception:
        pass
    try:
        return float(client.objectiveController.getStatus()["pixelsize"])
    except Exception:
        return 1.0


cal = None
if CAL_FILE.exists() and not RECALIBRATE:
    cal = smart.StageCameraCalibration.load(CAL_FILE)
    if cal.details.get("frame_shape") != [H, W]:
        print("cached calibration was made at another frame size - recalibrating")
        cal = None
if cal is None:
    step_um = 0.15 * min(H, W) * pixel_size_guess()     # ~15 % of the field of view
    print(f"calibrating with {step_um:.1f} µm steps ...")
    cal = smart.StageCameraCalibration.measure(snap, move, step_um=step_um, settle_s=0.3)
    cal.save(CAL_FILE)
print(cal)
print("details:", {k: v for k, v in cal.details.items() if k != "frame_shape"})
FOV_UM = (H * cal.pixel_size_um, W * cal.pixel_size_um)
print(f"field of view ≈ {FOV_UM[1]:.0f} x {FOV_UM[0]:.0f} µm")

## 4 Tracking settings

Centre the object you want to follow in the live view before you run the loop.

In [ ]:
TRACK_MODE = "template"       # "template" or "blob"
TEMPLATE_SIZE_PX = 64         # side of the patch that is followed (template mode)
TEMPLATE_UPDATE = 0.2         # 0 = keep the first look, 1 = replace every frame (adapts to shape changes)
MIN_MATCH = 0.4               # normalised correlation below this counts as "lost"
BLOB = dict(min_area=20, dark_objects=False, k=3.0)   # blob mode, see smart.find_objects
GATE_PX = 100                 # blob mode: max jump between frames

DEADBAND_PX = 4               # do not move for offsets smaller than this
GAIN = 0.8                    # fraction of the measured offset corrected per step (<1 = smoother)
MAX_STEP_UM = 0.5 * min(FOV_UM)
BOX_HALF_WIDTH_UM = 3000
N_STEPS = 40
INTERVAL_S = 0.0              # extra pause per step
MAX_LOST = 3                  # consecutive misses before giving up
SIMULATE_MOTION_UM = 0.0      # e.g. 15 on the virtual microscope: random walk per step
DISPLAY_EVERY = 1

## 5 Locate the target

In [ ]:
def normalise(img):
    img = img.astype(np.float32)
    return (img - img.mean()) / (img.std() + 1e-6)


def locate_template(frame, template):
    """(dy, dx) offset of the best match from the frame centre, and its score"""
    res = cv2.matchTemplate(normalise(frame), normalise(template), cv2.TM_CCOEFF_NORMED)
    _, score, _, (x, y) = cv2.minMaxLoc(res)
    th, tw = template.shape
    cy, cx = y + (th - 1) / 2, x + (tw - 1) / 2
    return (cy - (frame.shape[0] - 1) / 2, cx - (frame.shape[1] - 1) / 2), float(score)


def locate_blob(frame, expected_offset):
    objs = smart.find_objects(frame, **BLOB)
    if not objs:
        return None, 0.0
    best = min(objs, key=lambda o: np.hypot(o["offset"][0] - expected_offset[0],
                                            o["offset"][1] - expected_offset[1]))
    dist = np.hypot(best["offset"][0] - expected_offset[0], best["offset"][1] - expected_offset[1])
    return (best["offset"], 1.0) if dist <= GATE_PX else (None, 0.0)


def crop_at(frame, offset, size):
    cy = int(round((frame.shape[0] - 1) / 2 + offset[0]))
    cx = int(round((frame.shape[1] - 1) / 2 + offset[1]))
    half = size // 2
    cy = np.clip(cy, half, frame.shape[0] - half)
    cx = np.clip(cx, half, frame.shape[1] - half)
    return frame[cy - half:cy + half, cx - half:cx + half].copy()


first = snap()
template = crop_at(first, (0.0, 0.0), TEMPLATE_SIZE_PX)
if TRACK_MODE == "blob":
    blobs = smart.find_objects(first, **BLOB)
    print(f"{len(blobs)} blobs; following the one nearest the centre:",
          min(blobs, key=lambda o: np.hypot(*o["offset"])) if blobs else None)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), gridspec_kw={"width_ratios": [3, 1]})
axes[0].imshow(first, cmap="gray")
axes[0].add_patch(plt.Rectangle(((W - TEMPLATE_SIZE_PX) / 2, (H - TEMPLATE_SIZE_PX) / 2),
                                TEMPLATE_SIZE_PX, TEMPLATE_SIZE_PX, fill=False, color="r"))
axes[1].imshow(template, cmap="gray"); axes[1].set_title("template")
for a in axes: a.axis("off")
plt.show()

## 6 Track

Left: the current frame, with the target (red) and the dead band around the centre. Right: the
measured object position (red) and the stage path (grey), both in µm.

In [ ]:
x0, y0 = smart.get_xy(client, STAGE)
box = smart.SafetyBox(x0, y0, BOX_HALF_WIDTH_UM, MAX_STEP_UM)
rng = np.random.default_rng(0)
track = []                  # dicts: t, stage x/y, offset, object x/y, score, move
expected = (0.0, 0.0)
lost = 0
handle = display(None, display_id=True)


def draw(frame, rec):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(frame, cmap="gray")
    axes[0].add_patch(plt.Circle(((W - 1) / 2, (H - 1) / 2), DEADBAND_PX, fill=False, color="c"))
    if rec.get("offset") is not None:
        axes[0].plot((W - 1) / 2 + rec["offset"][1], (H - 1) / 2 + rec["offset"][0], "r+", ms=18, mew=2)
    axes[0].set_title(f"step {len(track)}  score {rec.get('score', 0):.2f}")
    axes[0].axis("off")
    ok = [r for r in track if r.get("obj") is not None]
    axes[1].plot([r["stage"][0] for r in track], [r["stage"][1] for r in track], ".-", c="0.6", label="stage")
    if ok:
        axes[1].plot([r["obj"][0] for r in ok], [r["obj"][1] for r in ok], "r.-", label="object")
    axes[1].set_xlabel("X [µm]"); axes[1].set_ylabel("Y [µm]"); axes[1].set_aspect("equal", "datalim")
    axes[1].legend(loc="best"); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    handle.update(fig)
    plt.close(fig)


try:
    for step in range(N_STEPS):
        if SIMULATE_MOTION_UM:
            move(*rng.normal(0, SIMULATE_MOTION_UM, 2))
        frame = snap()
        sx, sy = smart.get_xy(client, STAGE)
        if TRACK_MODE == "template":
            offset, score = locate_template(frame, template)
            if score < MIN_MATCH:
                offset = None
        else:
            offset, score = locate_blob(frame, expected)
        rec = {"t": time.time(), "stage": (sx, sy), "offset": offset, "score": score, "obj": None}
        if offset is None:
            lost += 1
            track.append(rec)
            if lost > MAX_LOST:
                print(f"target lost for {lost} frames - stopping")
                break
            continue
        lost = 0
        ox, oy = cal.stage_offset_of_pixel(*offset)
        rec["obj"] = (sx + ox, sy + oy)
        if TRACK_MODE == "template" and TEMPLATE_UPDATE:
            template = ((1 - TEMPLATE_UPDATE) * template
                        + TEMPLATE_UPDATE * crop_at(frame, offset, TEMPLATE_SIZE_PX))
        if np.hypot(*offset) > DEADBAND_PX:
            mx, my = (GAIN * v for v in cal.move_to_center(*offset))
            if not box.contains(sx + mx, sy + my):
                print("object is leaving the safety box - stopping")
                track.append(rec)
                break
            mx, my, _ = box.limit(sx, sy, mx, my)       # caps the step at MAX_STEP_UM
            move(mx, my)
            rec["move"] = (mx, my)
            expected = (0.0, 0.0)          # we re-centred, expect it near the middle
        else:
            expected = offset
        track.append(rec)
        if step % DISPLAY_EVERY == 0:
            draw(frame, rec)
        time.sleep(INTERVAL_S)
except KeyboardInterrupt:
    print("stopped")

found = [r for r in track if r["obj"] is not None]
print(f"{len(found)}/{len(track)} frames with the target")
if found:
    obj = np.array([r["obj"] for r in found])
    path = np.sum(np.hypot(*np.diff(obj, axis=0).T)) if len(obj) > 1 else 0.0
    print(f"object travelled {path:.0f} µm, net {np.hypot(*(obj[-1] - obj[0])):.0f} µm")
    if SIMULATE_MOTION_UM:
        print(f"simulated motion: the object did not really move, so the scatter of its measured "
              f"position ({obj.std(axis=0).round(1)} µm std) is the tracking error")

## 7 Save and inspect the trajectory

Writes the track as CSV and plots the speed. The stage stays where tracking ended.

In [ ]:
out = WORK_DIR / f"track_{datetime.datetime.now():%Y-%m-%d_%H-%M-%S}.csv"
with open(out, "w") as f:
    f.write("t_s,stage_x_um,stage_y_um,offset_y_px,offset_x_px,object_x_um,object_y_um,score\n")
    t0 = track[0]["t"] if track else 0
    for r in track:
        oy, ox = r["offset"] if r["offset"] is not None else ("", "")
        objx, objy = r["obj"] if r["obj"] is not None else ("", "")
        f.write(f"{r['t'] - t0:.3f},{r['stage'][0]},{r['stage'][1]},{oy},{ox},{objx},{objy},{r['score']:.3f}\n")
print("saved", out)

if len(found) > 2:
    t = np.array([r["t"] for r in found]); obj = np.array([r["obj"] for r in found])
    speed = np.hypot(*np.diff(obj, axis=0).T) / np.diff(t)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    sc = axes[0].scatter(obj[:, 0], obj[:, 1], c=t - t[0], cmap="viridis", s=12)
    axes[0].set_aspect("equal", "datalim"); axes[0].set_xlabel("X [µm]"); axes[0].set_ylabel("Y [µm]")
    fig.colorbar(sc, ax=axes[0], label="time [s]")
    axes[1].plot(t[1:] - t[0], speed); axes[1].set_xlabel("time [s]"); axes[1].set_ylabel("speed [µm/s]")
    fig.tight_layout(); plt.show()